# Results and diagnostics

Reads saved scores to produce crop-level precision/recall/F1, coverage tables and confusion matrices. Shows full and matched comparisons without choosing a winner using the test set.

Run after preprocessing 03 has exported a complete feature snapshot. Existing snapshots can be used while collection continues. Only load trusted local model packages.

In [ ]:
from pathlib import Path
import sys

root = Path.cwd().resolve()
while not (root / "pyproject.toml").is_file():
    if root == root.parent:
        raise RuntimeError("Open this notebook inside the project repository")
    root = root.parent
if str(root / "src") not in sys.path:
    sys.path.insert(0, str(root / "src"))
from crop_pipeline.training import notebook_workflow as workflow


In [ ]:
scores_handoff = ""  # Optional scores.json; blank uses latest completed test run
report_handoff = workflow.report(root, scores_handoff or None)
print((report_handoff.parent / "results.md").read_text())
print("Tables:", report_handoff.parent)

## Figures

External-test macro-F1 beside validation, the training-majority baseline, per-crop F1 and confusion matrices. The 0.50 line is the agreed target. Test scores are never used to choose a model.

In [ ]:
from IPython.display import Image, display
import matplotlib.pyplot as plt
from crop_pipeline.visualization import workflow_figures as figures


def show_and_save(figure, folder, name):
    """Display inline and keep a PNG beside the run outputs for reports and slides."""
    print("Saved figure:", figures.save_figure(figure, folder, name))
    display(figure)
    plt.close(figure)

import json
from pathlib import Path

reports = report_handoff.parent.parent
test_scores = json.loads((reports / "test_scores.json").read_text())
baseline = figures.baseline_macro_f1(json.loads((reports / "majority_baseline.json").read_text()))
models_handoff = Path(json.loads((reports.parent / "scores.json").read_text())["models_handoff"])
validation_scores = figures.load_validation_scores(models_handoff.parent / "models.json")
figure_folder = reports / "figures"
show_and_save(figures.draw_macro_f1(test_scores, "Validation vs external-test macro-F1", baseline=baseline,
                                    comparison=validation_scores), figure_folder, "test_macro_f1")
show_and_save(figures.draw_per_crop_f1(test_scores, "Per-crop external-test F1 (target ≥ 0.50)"),
              figure_folder, "test_per_crop_f1")
show_and_save(figures.draw_confusion_matrices(test_scores, "External-test confusion matrices (parcels; shade = row share)"),
              figure_folder, "test_confusion_matrices")